# 03c — Instructor Features

Different row structure from 03a/03b — one row per **instructor × course** pair.
Used only for model_prof.

**Input:**  `data/processed/sfu_clean.db` (train rows only)
**Output:** `data/processed/03c_instructor.csv`

**Output columns:**
`instructor, ml_course_id, instructor_n_sections_total, instructor_n_terms_taught,
instructor_summer_rate, instructor_n_times_this_course,
instructor_taught_this_course, instructor_dept_match`

## 0. Setup

In [12]:
import sqlite3
import pandas as pd
import numpy as np
from pathlib import Path

CLEAN_DB = Path('../data/processed/sfu_clean.db')
OUT_PATH = Path('../data/processed')

assert CLEAN_DB.exists(), f'not found: {CLEAN_DB}'
print('ready')

ready


## 1. Load train rows with instructor

In [13]:
conn = sqlite3.connect(CLEAN_DB)
df   = pd.read_sql('SELECT * FROM offerings', conn)
conn.close()

# same train boundary as 03a and 03b
TRAIN_YEAR = 2024
TRAIN_TERM = 3

train_mask = (
    (df['year'] < TRAIN_YEAR) |
    ((df['year'] == TRAIN_YEAR) & (df['term_order'] <= TRAIN_TERM))
)
train_raw = df[train_mask].copy()

# keep only rows with a real instructor name
instr = train_raw[
    train_raw['instructor'].notna() &
    (train_raw['instructor'].str.strip() != '')
].copy()

print(f'train rows total:           {len(train_raw):,}')
print(f'train rows with instructor: {len(instr):,}')
print(f'unique instructors:         {instr["instructor"].nunique():,}')
print(f'unique courses taught:      {instr["ml_course_id"].nunique():,}')

train rows total:           27,327
train rows with instructor: 22,751
unique instructors:         2,965
unique courses taught:      2,874


## 2. Per-instructor overall stats

In [14]:
instr_overall = (
    instr
    .groupby('instructor')
    .agg(
        instructor_n_sections_total = ('offering_id', 'count'),
        instructor_n_terms_taught   = ('ml_term_id',  'nunique'),
        _n_summer                   = ('term', lambda s: (s == 'summer').sum()),
        _n_fall                     = ('term', lambda s: (s == 'fall').sum()),
        _n_spring                   = ('term', lambda s: (s == 'spring').sum()),
    )
    .reset_index()
)

instr_overall['instructor_summer_rate'] = (
    instr_overall['_n_summer'] / instr_overall['instructor_n_sections_total']
).round(3)

instr_overall['instructor_fall_rate'] = (
    instr_overall['_n_fall'] / instr_overall['instructor_n_sections_total']
).round(3)

instr_overall['instructor_spring_rate'] = (
    instr_overall['_n_spring'] / instr_overall['instructor_n_sections_total']
).round(3)

instr_overall = instr_overall.drop(columns=['_n_summer','_n_fall','_n_spring'])

print(f'instr_overall: {len(instr_overall):,} instructors  x  {instr_overall.shape[1]} cols')
instr_overall.head(3)

instr_overall: 2,965 instructors  x  6 cols


,instructor,instructor_n_sections_total,instructor_n_terms_taught,instructor_summer_rate,instructor_fall_rate,instructor_spring_rate
0,"(Farrell) MacRae, Tracy; Philipp-Muller, Aviva",1,1,0.0,0.0,1.0
1,"ACKAH-ARTHUR, JEMIMA",1,1,1.0,0.0,0.0
2,"Abadi, Reza",1,1,0.0,1.0,0.0


## 3. Primary department per instructor

The department they taught in most often across all train sections.

In [15]:
instr_dept = (
    instr
    .groupby(['instructor','dept_code'])
    .size()
    .reset_index(name='n')
    .sort_values('n', ascending=False)
    .drop_duplicates(subset='instructor')   # keep top dept per instructor
    .rename(columns={'dept_code': 'primary_dept'})
    [['instructor','primary_dept']]
)

instr_overall = instr_overall.merge(instr_dept, on='instructor', how='left')

print(f'instr_overall with dept: {len(instr_overall):,} rows')
instr_overall.head(3)

instr_overall with dept: 2,965 rows


,instructor,instructor_n_sections_total,instructor_n_terms_taught,instructor_summer_rate,instructor_fall_rate,instructor_spring_rate,primary_dept
0,"(Farrell) MacRae, Tracy; Philipp-Muller, Aviva",1,1,0.0,0.0,1.0,BUS
1,"ACKAH-ARTHUR, JEMIMA",1,1,1.0,0.0,0.0,IS
2,"Abadi, Reza",1,1,0.0,1.0,0.0,MSE


## 4. Per instructor × course stats

In [16]:
instr_per_course = (
    instr
    .groupby(['instructor','ml_course_id'])
    .agg(
        instructor_n_times_this_course = ('offering_id', 'count'),
    )
    .reset_index()
)

# if this row exists, the instructor taught the course — always 1
instr_per_course['instructor_taught_this_course'] = 1

print(f'instr_per_course: {len(instr_per_course):,} instructor-course pairs')
instr_per_course.head(3)

instr_per_course: 9,435 instructor-course pairs


,instructor,ml_course_id,instructor_n_times_this_course,instructor_taught_this_course
0,"(Farrell) MacRae, Tracy; Philipp-Muller, Aviva",231,1,1
1,"ACKAH-ARTHUR, JEMIMA",1331,1,1
2,"Abadi, Reza",2816,1,1


## 5. Merge and derive dept_match

In [17]:
# join overall stats onto per-course table
instructor_features = instr_per_course.merge(instr_overall, on='instructor', how='left')

# get the dept for each course
course_dept = (
    instr[['ml_course_id','dept_code']]
    .drop_duplicates(subset='ml_course_id')
)
instructor_features = instructor_features.merge(course_dept, on='ml_course_id', how='left')

# dept match: does instructor's primary dept match this course's dept?
instructor_features['instructor_dept_match'] = (
    instructor_features['primary_dept'] == instructor_features['dept_code']
).astype(int)

# drop intermediate columns
instructor_features = instructor_features.drop(columns=['primary_dept','dept_code'])

print(f'instructor_features: {instructor_features.shape}')
print(f'columns: {list(instructor_features.columns)}')
instructor_features.head(3)

instructor_features: (9435, 10)
columns: ['instructor', 'ml_course_id', 'instructor_n_times_this_course', 'instructor_taught_this_course', 'instructor_n_sections_total', 'instructor_n_terms_taught', 'instructor_summer_rate', 'instructor_fall_rate', 'instructor_spring_rate', 'instructor_dept_match']


,instructor,ml_course_id,instructor_n_times_this_course,instructor_taught_this_course,instructor_n_sections_total,instructor_n_terms_taught,instructor_summer_rate,instructor_fall_rate,instructor_spring_rate,instructor_dept_match
0,"(Farrell) MacRae, Tracy; Philipp-Muller, Aviva",231,1,1,1,1,0.0,0.0,1.0,1
1,"ACKAH-ARTHUR, JEMIMA",1331,1,1,1,1,1.0,0.0,0.0,1
2,"Abadi, Reza",2816,1,1,1,1,0.0,1.0,0.0,1


## 6. Verify

In [18]:
expected_cols = [
    'instructor','ml_course_id',
    'instructor_n_sections_total','instructor_n_terms_taught',
    'instructor_summer_rate','instructor_n_times_this_course',
    'instructor_taught_this_course','instructor_dept_match'
]
missing = [c for c in expected_cols if c not in instructor_features.columns]
extra   = [c for c in instructor_features.columns if c not in expected_cols]
print(f'missing cols: {missing if missing else "none"}')
print(f'extra cols:   {extra if extra else "none"}')
print()
nulls = instructor_features.isnull().sum()
print('null counts:')
print(nulls[nulls > 0] if nulls.any() else 'none')

missing cols: none
extra cols:   ['instructor_fall_rate', 'instructor_spring_rate']

null counts:
none


In [19]:
# spot check: find a known instructor and see their row
# pick whoever taught the most sections
top_instr = instr_overall.nlargest(1, 'instructor_n_sections_total')['instructor'].values[0]
print(f'most active instructor: {top_instr}')
print()
sample = instructor_features[instructor_features['instructor'] == top_instr]
print(f'courses taught: {len(sample)}')
sample.head(5)

most active instructor: Erickson, Natalie

courses taught: 24


,instructor,ml_course_id,instructor_n_times_this_course,instructor_taught_this_course,instructor_n_sections_total,instructor_n_terms_taught,instructor_summer_rate,instructor_fall_rate,instructor_spring_rate,instructor_dept_match
2689,"Erickson, Natalie",3,25,1,286,15,0.371,0.311,0.318,0
2690,"Erickson, Natalie",4,21,1,286,15,0.371,0.311,0.318,0
2691,"Erickson, Natalie",9,20,1,286,15,0.371,0.311,0.318,0
2692,"Erickson, Natalie",10,10,1,286,15,0.371,0.311,0.318,0
2693,"Erickson, Natalie",11,3,1,286,15,0.371,0.311,0.318,0


In [20]:
# distribution checks
print('instructor_dept_match distribution:')
print(instructor_features['instructor_dept_match'].value_counts())
print()
print('instructor_summer_rate distribution:')
print(instructor_features['instructor_summer_rate'].describe().round(3))
print()
print('instructor_n_times_this_course distribution:')
print(instructor_features['instructor_n_times_this_course'].describe().round(1))

instructor_dept_match distribution:
instructor_dept_match
1    8884
0     551
Name: count, dtype: int64

instructor_summer_rate distribution:
count    9435.000
mean        0.220
std         0.245
min         0.000
25%         0.000
50%         0.167
75%         0.343
max         1.000
Name: instructor_summer_rate, dtype: float64

instructor_n_times_this_course distribution:
count    9435.0
mean        2.4
std         2.7
min         1.0
25%         1.0
50%         1.0
75%         3.0
max        37.0
Name: instructor_n_times_this_course, dtype: float64


## 7. Save

In [21]:
instructor_features.to_csv(OUT_PATH / '03c_instructor.csv', index=False)
print(f'saved 03c_instructor.csv: {len(instructor_features):,} rows  x  {instructor_features.shape[1]} cols')

saved 03c_instructor.csv: 9,435 rows  x  10 cols


In [22]:
# read back to verify
ins = pd.read_csv(OUT_PATH / '03c_instructor.csv')
assert ins.shape == instructor_features.shape
print('verified')
print(f'columns: {list(ins.columns)}')

verified
columns: ['instructor', 'ml_course_id', 'instructor_n_times_this_course', 'instructor_taught_this_course', 'instructor_n_sections_total', 'instructor_n_terms_taught', 'instructor_summer_rate', 'instructor_fall_rate', 'instructor_spring_rate', 'instructor_dept_match']
